# Preprocesamiento y Splits

Carga el catalogo limpio, codifica las columnas categoricas y genera los splits
de entrenamiento/prueba para los dos problemas:
- **Clasificacion binaria** (`is_hit`): Arbol de Decision y SVM
- **Clasificacion multiclase** (`imdb_clase`): Red Neuronal (MLP)

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import sys

sys.path.append('..')

from src.preprocessing import (
    codificar_categoricas,
    preparar_datos,
    COLUMNAS_NUMERICAS,
    COLUMNAS_CATEGORICAS,
)

## 1. Cargar datos

In [2]:
df = pd.read_csv('../data/processed/catalogo_procesado.csv')
print(f'Filas cargadas: {len(df)}')
print(f'Columnas disponibles: {df.shape[1]}')

# Verificar que las columnas necesarias esten presentes
cols_esperadas = COLUMNAS_NUMERICAS + ['is_hit', 'imdb_clase', 'rotten_tomatoes_score']
faltantes = [c for c in cols_esperadas if c not in df.columns]
if faltantes:
    print(f'Columnas faltantes: {faltantes}')
else:
    print('Todas las columnas estan presentes.')

# Verificar nulos en las features numericas
nulos = df[COLUMNAS_NUMERICAS].isnull().sum()
print('\nNulos en features numericas:')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Ninguno')

Filas cargadas: 15000
Columnas disponibles: 38
Todas las columnas estan presentes.

Nulos en features numericas:
Ninguno


## 2. Codificar columnas categoricas

`LabelEncoder` convierte texto a numero entero. Se guardan los encoders para reutilizarlos en el dashboard.

In [3]:
df, encoders = codificar_categoricas(df, COLUMNAS_CATEGORICAS)

for col in COLUMNAS_CATEGORICAS:
    n = df[col].nunique()
    print(f'{col:10s} - {col}_cod  ({n} categorias)')

platform   - platform_cod  (10 categorias)
primary_genre - primary_genre_cod  (21 categorias)
rating     - rating_cod  (12 categorias)


## 3. Definir features para cada modelo

In [4]:
# Features para clasificacion binaria (Arbol de Decision y SVM)
# votos_log se incluye: es un predictor legitimo (engagement externo),
# no es el target (hours_watched) sino una variable distinta (conteo de votos IMDb)
FEATURES_CLF = COLUMNAS_NUMERICAS + ['platform_cod', 'primary_genre_cod', 'rating_cod', 'rotten_tomatoes_score']

# Features para la Red Neuronal (imdb_clase)
# Excluimos rotten_tomatoes_score: mide calidad similar al target imdb_clase
FEATURES_NN = COLUMNAS_NUMERICAS + ['platform_cod', 'primary_genre_cod', 'rating_cod']

print(f'Features CLF (is_hit):     {len(FEATURES_CLF)} columnas')
print(f'Features NN  (imdb_clase): {len(FEATURES_NN)} columnas')
print()
print('Columnas CLF:', FEATURES_CLF)

Features CLF (is_hit):     17 columnas
Features NN  (imdb_clase): 16 columnas

Columnas CLF: ['release_year', 'antiguedad', 'duracion_total', 'votos_log', 'presupuesto_log', 'awards_won', 'cantidad_generos', 'tamanio_elenco', 'es_pelicula', 'es_reciente', 'tiene_presupuesto', 'mes_agregado', 'dias_en_plataforma', 'platform_cod', 'primary_genre_cod', 'rating_cod', 'rotten_tomatoes_score']


## 4. Preparar datos - Clasificacion binaria (`is_hit`)

Split 80/20 estratificado + `StandardScaler`.

In [5]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf, scaler_clf = preparar_datos(
    df, 'is_hit', FEATURES_CLF
)

print(f'Train: {X_train_clf.shape}   Test: {X_test_clf.shape}')
print()
print('Balance is_hit en train:')
print(y_train_clf.value_counts().rename({0: 'No Hit', 1: 'Hit'}).to_string())

Train: (12000, 17)   Test: (3000, 17)

Balance is_hit en train:
is_hit
Hit       6002
No Hit    5998


## 5. Preparar datos - Clasificacion multiclase (`imdb_clase`)

Convertimos las etiquetas de texto a numeros: `Bajo=0`, `Medio=1`, `Alto=2`.

In [6]:
# Convertir etiquetas de texto a numeros
df['imdb_clase_cod'] = df['imdb_clase'].map({'Bajo': 0, 'Medio': 1, 'Alto': 2})

nulos_imdb = df['imdb_clase_cod'].isna().sum()
print(f'Filas con imdb_clase nulo: {nulos_imdb}')

X_train_nn, X_test_nn, y_train_nn, y_test_nn, scaler_nn = preparar_datos(
    df, 'imdb_clase_cod', FEATURES_NN
)

print(f'Train: {X_train_nn.shape}   Test: {X_test_nn.shape}')
print()
print('Balance imdb_clase en train:')
conteo = y_train_nn.value_counts().sort_index()
for i, nombre in enumerate(['Bajo', 'Medio', 'Alto']):
    print(f'{nombre}: {conteo.get(i, 0)}')

Filas con imdb_clase nulo: 0
Train: (12000, 16)   Test: (3000, 16)

Balance imdb_clase en train:
Bajo: 3320
Medio: 6014
Alto: 2666


## 6. Guardar splits como CSV

In [7]:
# Splits para clasificacion binaria
pd.DataFrame(X_train_clf, columns=FEATURES_CLF).to_csv('../data/processed/X_train_clf.csv', index=False)
pd.DataFrame(X_test_clf,  columns=FEATURES_CLF).to_csv('../data/processed/X_test_clf.csv',  index=False)
y_train_clf.to_csv('../data/processed/y_train_clf.csv', index=False)
y_test_clf.to_csv( '../data/processed/y_test_clf.csv',  index=False)

# Splits para clasificacion multiclase
pd.DataFrame(X_train_nn, columns=FEATURES_NN).to_csv('../data/processed/X_train_nn.csv', index=False)
pd.DataFrame(X_test_nn,  columns=FEATURES_NN).to_csv('../data/processed/X_test_nn.csv',  index=False)
y_train_nn.to_csv('../data/processed/y_train_nn.csv', index=False)
y_test_nn.to_csv( '../data/processed/y_test_nn.csv',  index=False)

print('CSVs guardados en data/processed/')

CSVs guardados en data/processed/


## 7. Guardar scalers y encoders

Se guardan con `joblib` para reutilizarlos en el dashboard sin reentrenar.

In [8]:
joblib.dump(scaler_clf, '../models/scaler_clf.joblib')
joblib.dump(scaler_nn,  '../models/scaler_nn.joblib')
joblib.dump(encoders,   '../models/encoders.joblib')

# Guardar lista de features para reutilizar en el dashboard
config = {
    'FEATURES_CLF': FEATURES_CLF,
    'FEATURES_NN': FEATURES_NN,
}
with open('../models/features_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Archivos guardados en models/')
print('scaler_clf.joblib')
print('scaler_nn.joblib')
print('encoders.joblib')
print('features_config.json')

Archivos guardados en models/
scaler_clf.joblib
scaler_nn.joblib
encoders.joblib
features_config.json


## 8. Verificacion final

In [9]:
print(f'X_train_clf : {X_train_clf.shape}')
print(f'X_test_clf  : {X_test_clf.shape}')
print(f'X_train_nn  : {X_train_nn.shape}')
print(f'X_test_nn   : {X_test_nn.shape}')


X_train_clf : (12000, 17)
X_test_clf  : (3000, 17)
X_train_nn  : (12000, 16)
X_test_nn   : (3000, 16)
